# 01 - Data Loading
Load CMEMS oceanographic data and AIS fishing effort for the West Philippine Sea ConvLSTM pipeline.

**Outputs:** `physics_raw_region.nc`, `bgc_raw_region.nc`, `ais_raw_region.parquet`

Run this notebook once to cache regional data, then proceed to `02_preprocessing.ipynb`.

In [ ]:
# CELL: Install all pipeline dependencies at once
!pip install -q xarray netCDF4 pandas numpy matplotlib pyarrow scikit-learn scipy


In [ ]:
# CELL: Shared imports, Google Drive mount, and unified pipeline CONFIG
# =========================================================
# This cell initializes all libraries and the complete CONFIG.
# Subsequent cells use these variables directly with NO redundant imports/re-definitions.
# =========================================================
import os
import glob
import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.stats import wasserstein_distance
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score
from matplotlib.colors import LinearSegmentedColormap

# Google Colab Drive Mount
from google.colab import drive
drive.mount('/content/drive')

# TensorFlow imports for deep learning (used in training & evaluation)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Conv2D, BatchNormalization, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Complete, single CONFIG dictionary for the entire pipeline
CONFIG = {
    # ------------------------------------------------------------------
    # Spatial
    # ------------------------------------------------------------------
    # Broad regional bbox: used in Data Loading to cache raw CMEMS/AIS.
    # Wide enough to allow experimenting with different sub-regions.
    'bbox_regional': {
        'lat_min': 0,   'lat_max': 30,
        'lon_min': 110, 'lon_max': 140,
    },
    # WPS model target bbox: used from Preprocessing onward.
    # Narrow domain improves class balance and F1 for ConvLSTM training.
    'bbox_model': {
        'lat_min': 10,  'lat_max': 20,
        'lon_min': 114, 'lon_max': 120,
    },

    # ------------------------------------------------------------------
    # Temporal
    # ------------------------------------------------------------------
    # Full date range for raw CMEMS/AIS load.
    'date_full': {'start': '2014-01-01', 'end': '2024-12-31'},
    # ConvLSTM training window. Starts 2019 to use denser AIS coverage.
    'date_model': {'start': '2019-01-01', 'end': '2024-12-31'},

    # ------------------------------------------------------------------
    # Paths & Files
    # ------------------------------------------------------------------
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        # Source CMEMS NetCDF files (placed in data_dir by the user)
        'physics_w_nc':   'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc':  'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':     'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        
        # Data Loading outputs -> Preprocessing inputs
        'physics_nc':     'physics_raw_region.nc',
        'bgc_nc':         'bgc_raw_region.nc',
        'ais_parquet':    'ais_raw_region.parquet',
        'ais_csv_gz':     'ais_raw_region.csv.gz',
        
        # Preprocessing outputs -> Training / Evaluation inputs
        'ais_gridded_nc': 'ais_fishing_effort_gridded.nc',
        'preprocessed_nc':'preprocessed_features.nc',
        
        # Model Training outputs -> Evaluation / Visualization inputs
        'model_keras':    'convlstm_model.keras',
        'best_model':     'best_model.keras',
        'X_test_npy':     'X_test.npy',
        'y_test_npy':     'y_test.npy',
        'history_json':   'training_history.json',
        'summary_json':   'data_summary.json',
        
        # Evaluation outputs -> Visualization inputs
        'predictions_npy':'predictions.npy',
        'eval_csv':       'evaluation_results.csv',
    },

    # ------------------------------------------------------------------
    # AIS Configuration
    # ------------------------------------------------------------------
    # Columns to read from each GFW monthly CSV (skips vessel metadata)
    'ais_use_cols': ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],

    # ------------------------------------------------------------------
    # Depth Selection for CMEMS Variables
    # ------------------------------------------------------------------
    'physics_surface_depth': 0.49,   # metres, nearest-neighbour selection
    'bgc_depth_range': (0.51, 5.14), # metres, averaged over this range

    # ------------------------------------------------------------------
    # Preprocessing Configuration
    # ------------------------------------------------------------------
    # Normalization method for oceanographic channels: 'minmax' or 'zscore'
    'norm_method': 'minmax',
    # xarray/pandas resample frequency ('1ME' replaces deprecated '1M')
    'resample_freq': '1ME',

    # ------------------------------------------------------------------
    # Model / Training Hyperparameters
    # ------------------------------------------------------------------
    'seq_len':    3,     # months of input context fed to ConvLSTM
    'pred_len':   1,     # months ahead to predict
    'n_channels': 7,     # sst, ssh, vo, uo, chl, nppv, fishing_effort
    'train_frac': 0.70,
    'val_frac':   0.15,
    # test_frac = 1 - 0.70 - 0.15 = 0.15
    'epochs':     50,
    'batch_size': 8,
    'patience':   10,

    # ------------------------------------------------------------------
    # Evaluation Configuration
    # ------------------------------------------------------------------
    'f1_threshold': 0.5,   # binarization threshold for F1 score in evaluation
}

# Global Convenience Aliases
DATA_DIR   = CONFIG['data_dir']
f          = CONFIG['files']
bbox_r     = CONFIG['bbox_regional']
dates_full = CONFIG['date_full']

print(f'DATA_DIR      : {DATA_DIR}')
print(f'Regional bbox : {bbox_r}')
print(f'Full dates    : {dates_full}')
print("Unified imports & CONFIG pipeline setup complete!")


In [ ]:
# CELL: load raw CMEMS NetCDF datasets from Drive
f = CONFIG['files']

print('Loading CMEMS Global Ocean Physics Reanalysis (Wind Velocities)...')
w_physics_ds = xr.open_dataset(DATA_DIR + f['physics_w_nc'])
print(f'  Velocity variables : {list(w_physics_ds.data_vars)}')
print(f'  Shape              : {dict(w_physics_ds.dims)}')

print('Loading CMEMS Global Ocean Physics Reanalysis (Height & Temp)...')
physics_ds = xr.open_dataset(DATA_DIR + f['physics_ht_nc'])
print(f'  Variables : {list(physics_ds.data_vars)}')
print(f'  Shape     : {dict(physics_ds.dims)}')

print('Loading CMEMS Global Ocean Biogeochemistry Hindcast...')
bgc_ds = xr.open_dataset(DATA_DIR + f['bgc_src_nc'])
print(f'  Variables : {list(bgc_ds.data_vars)}')
print(f'  Shape     : {dict(bgc_ds.dims)}')

In [ ]:
# CELL: load AIS fishing effort CSVs (flat folder, regional bbox-filtered)
# Reads only ais_use_cols; skips files outside the date range by filename;
# filters rows to the regional bbox before concat -- never loads global data.
r  = CONFIG['bbox_regional']
dt = CONFIG['date_full']
AIS_ROOT = DATA_DIR + 'ais_fishing/'

def load_ais_filtered(ais_root, date_start, date_end,
                      lat_min, lat_max, lon_min, lon_max, use_cols):
    '''Load GFW AIS monthly CSVs filtered to a bounding box and date range.'''
    start  = pd.Timestamp(date_start)
    end    = pd.Timestamp(date_end)
    chunks = []

    # All CSVs are flat in one folder -- no year subfolders
    all_files = sorted(glob.glob(os.path.join(ais_root, '*.csv')))
    print(f'Total CSVs found in folder: {len(all_files)}')

    for fp in all_files:
        fname = os.path.basename(fp)
        # Extract date from filename: fleet-monthly-csvs-10-v3-YYYY-MM-DD.csv
        try:
            file_date = pd.Timestamp(fname[-14:-4])
        except Exception:
            continue
        # Skip files outside date range without opening them
        if not (start <= file_date <= end):
            continue

        try:
            df = pd.read_csv(
                fp,
                usecols=use_cols,
                dtype={
                    'cell_ll_lat':   'float32',
                    'cell_ll_lon':   'float32',
                    'fishing_hours': 'float32',
                }
            )
            # Filter to bbox immediately (drops ~98% of rows)
            mask = (
                (df['cell_ll_lat'] >= lat_min) & (df['cell_ll_lat'] <  lat_max) &
                (df['cell_ll_lon'] >= lon_min) & (df['cell_ll_lon'] <  lon_max)
            )
            filtered = df[mask]
            if not filtered.empty:
                chunks.append(filtered)
                n_rows = len(filtered)
                print(f'  {fname[-14:-4]}: {n_rows:,} rows in bbox')
        except Exception as e:
            print(f'  Skipped {fname}: {e}')

    if not chunks:
        raise ValueError('No AIS data found for the given bbox / date range!')

    ais_df = pd.concat(chunks, ignore_index=True)
    ais_df['date']       = pd.to_datetime(ais_df['date'])
    ais_df['year_month'] = ais_df['date'].dt.to_period('M')
    return ais_df


start_yr = dt['start'][:4]
end_yr   = dt['end'][:4]
print(f'Loading AIS CSVs ({start_yr}-{end_yr}, regional bbox)...')
ais_df = load_ais_filtered(
    AIS_ROOT,
    dt['start'], dt['end'],
    r['lat_min'], r['lat_max'], r['lon_min'], r['lon_max'],
    CONFIG['ais_use_cols']
)

n_records    = len(ais_df)
ais_min_date = ais_df['date'].min().date()
ais_max_date = ais_df['date'].max().date()
n_months     = ais_df['year_month'].nunique()
print(f'AIS records in regional bbox : {n_records:,}')
print(f'Date range                   : {ais_min_date} to {ais_max_date}')
print(f'Unique months                : {n_months}')

In [ ]:
# CELL: merge physics datasets into a single xarray Dataset
# Ensure consistent dimension order before merging
w_physics_ds = w_physics_ds.transpose('time', 'depth', 'latitude', 'longitude')
physics_ds   = physics_ds.transpose('time', 'depth', 'latitude', 'longitude')

print('Merging physics datasets (uo, vo, zos, thetao)...')
physics_ds = xr.merge([w_physics_ds, physics_ds])

combined_vars  = list(physics_ds.data_vars)
combined_shape = dict(physics_ds.dims)
print(f'Combined variables : {combined_vars}')
print(f'Combined shape     : {combined_shape}')

In [ ]:
# CELL: time range verification -- confirm all three sources share the configured span
print('=== Time Range Verification ===')
phys_min = physics_ds.time.min().values
phys_max = physics_ds.time.max().values
bgc_min  = bgc_ds.time.min().values
bgc_max  = bgc_ds.time.max().values
ais_min  = ais_df['date'].min()
ais_max  = ais_df['date'].max()

print(f'Physics data : {phys_min} to {phys_max}')
print(f'BGC data     : {bgc_min} to {bgc_max}')
print(f'AIS data     : {ais_min} to {ais_max}')

expected_start = pd.Timestamp(CONFIG['date_full']['start'])
assert pd.Timestamp(str(phys_min)) >= expected_start, 'Physics start out of configured range!'
assert pd.Timestamp(str(bgc_min))  >= expected_start, 'BGC start out of configured range!'
assert ais_min                      >= expected_start, 'AIS start out of configured range!'
print('All three sources cover the configured date range.')

In [ ]:
# CELL: save outputs -- regional NetCDFs + AIS Parquet for Notebook 02
# These files are the sole inputs to 02_preprocessing.ipynb.
f = CONFIG['files']

physics_nc_name  = f['physics_nc']
bgc_nc_name      = f['bgc_nc']
ais_parquet_name = f['ais_parquet']
ais_csv_gz_name  = f['ais_csv_gz']

# 1. Save merged raw physics and BGC regional datasets
physics_ds.to_netcdf(DATA_DIR + physics_nc_name)
bgc_ds.to_netcdf(DATA_DIR + bgc_nc_name)
print(f'Saved physics : {physics_nc_name}')
print(f'Saved BGC     : {bgc_nc_name}')

# 2. Save regional AIS DataFrame as Parquet (fast read, preserves dtypes)
try:
    ais_df.to_parquet(DATA_DIR + ais_parquet_name, index=False)
    print(f'Saved AIS     : {ais_parquet_name}')
except Exception:
    # Fallback to compressed CSV if pyarrow/fastparquet not available
    ais_df.to_csv(DATA_DIR + ais_csv_gz_name, compression='gzip', index=False)
    print(f'Saved AIS     : {ais_csv_gz_name} (compressed CSV fallback)')

print('Notebook 01 complete. Run 02_preprocessing.ipynb next.')


---
## Merged Section: 02_preprocessing.ipynb
---


# 02 - Preprocessing
Subset, regrid, normalize CMEMS and AIS data to the WPS ConvLSTM model domain.

**Inputs (from Notebook 01):** `physics_raw_region.nc`, `bgc_raw_region.nc`, `ais_raw_region.parquet`

**Outputs:** `preprocessed_features.nc` (7 channels, 41x25 grid, monthly)

In [ ]:
# CELL: Setup Notebook 02 local variables and convenience aliases
# CONFIG and common libraries are imported once in the first section.
bbox_m   = CONFIG['bbox_model']
dates_m  = CONFIG['date_model']
print(f'DATA_DIR   : {DATA_DIR}')
print(f'Model bbox : {bbox_m}')
print(f'Model dates: {dates_m}')


## STEP 1: Load and Subset Datasets

In [ ]:
# CELL: load regional datasets from Notebook 01, subset to WPS model bbox and dates
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
f  = CONFIG['files']

# Load the pre-merged regional datasets saved by Notebook 01
physics_ds = xr.open_dataset(DATA_DIR + f['physics_nc'])
bgc_ds     = xr.open_dataset(DATA_DIR + f['bgc_nc'])

phys_depths = physics_ds.depth.values[:5]
bgc_depths  = bgc_ds.depth.values[:5]
print(f'Available physics depths : {phys_depths}')
print(f'Available BGC depths     : {bgc_depths}')

# Physics: select nearest surface depth, then subset to WPS model bbox and dates
surf_depth = CONFIG['physics_surface_depth']
physics_subset = physics_ds.sel(
    depth=surf_depth, method='nearest'
).sel(
    latitude=slice(bm['lat_min'], bm['lat_max']),
    longitude=slice(bm['lon_min'], bm['lon_max']),
    time=slice(dm['start'], dm['end'])
)

# BGC: average over near-surface depths, then subset to WPS model bbox and dates
d_lo, d_hi = CONFIG['bgc_depth_range']
bgc_subset = bgc_ds.sel(
    depth=slice(d_lo, d_hi),
    latitude=slice(bm['lat_min'], bm['lat_max']),
    longitude=slice(bm['lon_min'], bm['lon_max']),
    time=slice(dm['start'], dm['end'])
).mean(dim='depth')

phys_dims = dict(physics_subset.dims)
bgc_dims  = dict(bgc_subset.dims)
print(f'Physics subset (0.083 deg) : {phys_dims}')
print(f'BGC subset    (0.25 deg)  : {bgc_dims}')

## STEP 2: Monthly Aggregation

In [ ]:
# CELL: resample to monthly means
# CMEMS data is already monthly, but resampling ensures consistent time-stamp
# labels and suppresses the FutureWarning from the deprecated '1M' string.
# CONFIG['resample_freq'] = '1ME' is the modern replacement.
freq = CONFIG['resample_freq']

physics_monthly = physics_subset.resample(time=freq).mean()
bgc_monthly     = bgc_subset.resample(time=freq).mean()

print(f'Monthly physics : {dict(physics_monthly.dims)}')
print(f'Monthly BGC     : {dict(bgc_monthly.dims)}')

## STEP 3: Define Target Grid and Regrid

In [ ]:
# CELL: regrid physics (0.083 deg) onto the BGC native grid (0.25 deg)
# Using BGC's native coordinates as the canonical target grid so all
# 7 ConvLSTM input channels share the same spatial resolution and extent.
target_lats = bgc_monthly.latitude.values
target_lons = bgc_monthly.longitude.values

n_lat     = len(target_lats)
n_lon     = len(target_lons)
lat_range = (target_lats.min(), target_lats.max())
lon_range = (target_lons.min(), target_lons.max())
print(f'Target grid : {n_lat} x {n_lon} at 0.25 deg resolution')
print(f'  Lat range : {lat_range[0]:.2f} to {lat_range[1]:.2f}')
print(f'  Lon range : {lon_range[0]:.2f} to {lon_range[1]:.2f}')

# Bilinear interpolation of physics to match BGC grid
physics_regrid = physics_monthly.interp(
    latitude=target_lats,
    longitude=target_lons,
    method='linear'
)

print(f'Physics regridded : {dict(physics_regrid.dims)}')

## STEP 4: Verify Alignment

In [ ]:
# CELL: assert grid and time alignment before downstream processing
assert np.allclose(physics_regrid.latitude,  bgc_monthly.latitude),  'Latitude mismatch!'
assert np.allclose(physics_regrid.longitude, bgc_monthly.longitude), 'Longitude mismatch!'
assert len(physics_regrid.time) == len(bgc_monthly.time),            'Time-step count mismatch!'

n_months = len(bgc_monthly.time)
n_pixels = n_lat * n_lon
print(f'All grids aligned -- {n_months} months at 0.25 deg resolution')
print(f'  Grid size : {n_lat} lat x {n_lon} lon = {n_pixels:,} pixels/month')

## STEP 5: Gap Filling

In [ ]:
# CELL: fill NaN gaps with linear temporal interpolation
def fill_gaps(data):
    '''Fill NaN gaps with linear interpolation along the time axis.
    extrapolate handles edge months with no valid neighbours.'''
    return data.interpolate_na(dim='time', method='linear', fill_value='extrapolate')

physics_filled = fill_gaps(physics_regrid)
bgc_filled     = fill_gaps(bgc_monthly)

# Report any remaining NaNs per variable after filling
for ds_name, ds in [('physics', physics_filled), ('bgc', bgc_filled)]:
    for var in ds.data_vars:
        n_nan  = int(np.isnan(ds[var].values).sum())
        status = 'OK' if n_nan == 0 else f'WARNING {n_nan} NaNs remain'
        print(f'  {ds_name}.{var}: {status}')

print('Gap filling complete.')

## STEP 6: Extract and Normalize Variables

In [ ]:
# CELL: extract all 6 oceanographic channels and normalize
# Normalization method is controlled by CONFIG['norm_method'].
# 'minmax' -> scales each variable to [0, 1] globally over the full time series.
# 'zscore' -> centres at mean=0, std=1 over the full time series.
def normalize(data, method=None):
    '''Normalize a DataArray using the method specified in CONFIG.'''
    if method is None:
        method = CONFIG['norm_method']
    if method == 'minmax':
        mn, mx = float(data.min()), float(data.max())
        return (data - mn) / (mx - mn) if mx > mn else data * 0
    elif method == 'zscore':
        return (data - data.mean()) / data.std()
    raise ValueError(f'Unknown normalization method: {method}')

# Extract variables from BGC dataset (biogeochemical)
chl  = normalize(bgc_filled['chl'])   # Chlorophyll-a
nppv = normalize(bgc_filled['nppv'])  # Net primary production

# Extract variables from Physics dataset
ssh = normalize(physics_filled['zos'])     # Sea surface height
sst = normalize(physics_filled['thetao'])  # Sea surface temperature
uo  = normalize(physics_filled['uo'])      # Eastward sea water velocity
vo  = normalize(physics_filled['vo'])      # Northward sea water velocity

method_used = CONFIG['norm_method']
print(f'Normalization complete (method={method_used})')
for var_name, arr in [('Chl', chl), ('NPPV', nppv), ('SSH', ssh),
                       ('SST', sst), ('UO',  uo),   ('VO',  vo)]:
    arr_min = float(arr.min())
    arr_max = float(arr.max())
    print(f'  {var_name:4s}: shape={arr.shape}  min={arr_min:.3f}  max={arr_max:.3f}')

## STEP 7: Process AIS Data

In [ ]:
# CELL: load pre-filtered AIS Parquet from Notebook 01, subset to WPS model bbox
# and date window, aggregate to 0.25 deg monthly grid, log-normalise, cache as NetCDF.
# NOTE: load_ais_filtered() is NOT redefined here -- Notebook 01 already filtered
# and saved ais_raw_region.parquet. We simply load and narrow that cache.
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
f  = CONFIG['files']

# --- 7a: Load from Notebook 01 output (Parquet preferred, CSV.gz fallback) ---
ais_parquet_name = f['ais_parquet']
ais_csv_gz_name  = f['ais_csv_gz']
print('Loading pre-filtered AIS data from Notebook 01...')
try:
    ais_df = pd.read_parquet(DATA_DIR + ais_parquet_name)
    print(f'  Loaded Parquet : {ais_parquet_name}')
except Exception:
    ais_df = pd.read_csv(DATA_DIR + ais_csv_gz_name)
    print(f'  Loaded CSV.gz fallback : {ais_csv_gz_name}')

# --- 7b: Narrow the regional AIS cache to the WPS model bbox ---
mask = (
    (ais_df['cell_ll_lat'] >= bm['lat_min']) & (ais_df['cell_ll_lat'] <  bm['lat_max']) &
    (ais_df['cell_ll_lon'] >= bm['lon_min']) & (ais_df['cell_ll_lon'] <  bm['lon_max'])
)
ais_df = ais_df[mask].copy()
ais_df['date']       = pd.to_datetime(ais_df['date'])
ais_df['year_month'] = ais_df['date'].dt.to_period('M')

# Further filter to the model date window (2019-2024)
date_mask = (
    (ais_df['date'] >= dm['start']) &
    (ais_df['date'] <= dm['end'])
)
ais_df = ais_df[date_mask]

n_records    = len(ais_df)
ais_min_date = ais_df['date'].min().date()
ais_max_date = ais_df['date'].max().date()
n_months_ais = ais_df['year_month'].nunique()
print(f'AIS records in WPS model bbox : {n_records:,}')
print(f'Date range                    : {ais_min_date} to {ais_max_date}')
print(f'Unique months                 : {n_months_ais}')


# --- 7c: Aggregate 0.1 deg GFW rows -> 0.25 deg monthly grid ---
def aggregate_ais_to_grid(ais_df, target_lats, target_lons):
    '''Bin AIS fishing_hours into the 0.25 deg target grid per month.
    GFW lower-left cell corners are shifted +0.05 to get cell centres.'''
    lat_c = ais_df['cell_ll_lat'].values + 0.05
    lon_c = ais_df['cell_ll_lon'].values + 0.05

    lat_idx = (np.searchsorted(target_lats, lat_c) - 1).clip(0, len(target_lats) - 1)
    lon_idx = (np.searchsorted(target_lons, lon_c) - 1).clip(0, len(target_lons) - 1)

    df = ais_df.copy()
    df['lat_idx'] = lat_idx
    df['lon_idx'] = lon_idx

    result = {}
    for period, grp in df.groupby('year_month'):
        grid = np.zeros((len(target_lats), len(target_lons)), dtype=np.float32)
        np.add.at(grid,
                  (grp['lat_idx'].values, grp['lon_idx'].values),
                  grp['fishing_hours'].values)
        result[str(period)] = grid
    return result


def ais_to_xarray(ais_grids, cmems_times, target_lats, target_lons):
    '''Align monthly AIS grids to the CMEMS time axis; missing months filled with 0.'''
    data = np.zeros((len(cmems_times), len(target_lats), len(target_lons)), dtype=np.float32)
    for i, t in enumerate(cmems_times):
        key = str(pd.Timestamp(t).to_period('M'))
        if key in ais_grids:
            data[i] = ais_grids[key]
    return xr.DataArray(
        data,
        dims=['time', 'latitude', 'longitude'],
        coords={'time': cmems_times, 'latitude': target_lats, 'longitude': target_lons},
        name='fishing_hours'
    )


def normalize_ais(da):
    '''Log1p then min-max normalization.
    Log1p compresses the heavy-tailed fishing effort distribution before scaling.
    Keeps zero-effort pixels at 0 after normalization.'''
    log_da = np.log1p(da)
    mn, mx = float(log_da.min()), float(log_da.max())
    return (log_da - mn) / (mx - mn) if mx > mn else log_da * 0


ais_grids      = aggregate_ais_to_grid(ais_df, target_lats, target_lons)
ais_da         = ais_to_xarray(ais_grids, bgc_monthly.time.values, target_lats, target_lons)
ais_normalized = normalize_ais(ais_da)

n_months_agg = len(ais_grids)
ais_da_sizes = dict(ais_da.sizes)
ais_norm_min = float(ais_normalized.min())
ais_norm_max = float(ais_normalized.max())
print(f'Months aggregated : {n_months_agg}')
print(f'AIS DataArray     : {ais_da_sizes}')
print(f'AIS normalized    : min={ais_norm_min:.3f}  max={ais_norm_max:.3f}')

# Cache gridded (un-normalised) AIS to Drive -- skips re-aggregation on restart
ais_gridded_nc_name = f['ais_gridded_nc']
ais_da.to_netcdf(DATA_DIR + ais_gridded_nc_name)
print(f'Saved AIS grid : {ais_gridded_nc_name}')

## STEP 8: Save Preprocessed Data

In [ ]:
# CELL: combine all 7 channels into one xarray Dataset and save to NetCDF
# This file is the sole input to 03_model_training.ipynb.
preprocessed = xr.Dataset({
    'chl':            chl,             # Chlorophyll-a              (BGC)
    'nppv':           nppv,            # Net primary production      (BGC)
    'ssh':            ssh,             # Sea surface height          (Physics)
    'sst':            sst,             # Sea surface temperature     (Physics)
    'uo':             uo,              # Eastward sea water velocity (Physics)
    'vo':             vo,              # Northward sea water velocity (Physics)
    'fishing_effort': ais_normalized,  # Log-normalised AIS fishing hours (AIS)
})

preprocessed_nc_name = CONFIG['files']['preprocessed_nc']
out_path = DATA_DIR + preprocessed_nc_name
preprocessed.to_netcdf(out_path)

final_dims = dict(preprocessed.dims)
final_vars = list(preprocessed.data_vars)
print(f'Preprocessed data saved : {preprocessed_nc_name}')
print(f'  Dimensions : {final_dims}')
print(f'  Variables  : {final_vars}')
print('Notebook 02 complete. Run 03_model_training.ipynb next.')


---
## Merged Section: 03_model_training.ipynb
---


# 03 - Model Training
ConvLSTM2D spatiotemporal model: 3-month sequences -> 1-month fishing ground prediction.

**Input (from Notebook 02):** `preprocessed_features.nc` (7 channels, 41x25 grid, monthly)

**Outputs:** `convlstm_model.keras`, `best_model.keras`, `X_test.npy`, `y_test.npy`, `training_history.json`, `data_summary.json`

In [ ]:
# CELL: load preprocessed features from Notebook 02
# Uses .sizes (not deprecated .dims mapping) to avoid FutureWarning in xarray >= 2023.x
SEQ_LEN    = CONFIG['seq_len']
PRED_LEN   = CONFIG['pred_len']
N_CHANNELS = CONFIG['n_channels']

features = xr.open_dataset(DATA_DIR + f['preprocessed_nc'])
n_months = features.sizes['time']
n_lat    = features.sizes['latitude']
n_lon    = features.sizes['longitude']

print(f'Preprocessed features:')
print(f'  time={n_months}, latitude={n_lat}, longitude={n_lon}')
print(f'  Variables: {list(features.data_vars)}')
# Expected: ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']


In [ ]:
# CELL: stack all 7 channels into a single NumPy array
# Channel order matches Full_ConvLSTM.py CHANNELS list: sst,ssh,vo,uo,chl,nppv,fishing_effort
X_stack = np.stack([
    features['sst'].values,            # Ch 0 - Sea surface temperature
    features['ssh'].values,            # Ch 1 - Sea surface height
    features['vo'].values,             # Ch 2 - Northward sea water velocity
    features['uo'].values,             # Ch 3 - Eastward sea water velocity
    features['chl'].values,            # Ch 4 - Chlorophyll-a
    features['nppv'].values,           # Ch 5 - Net primary production
    features['fishing_effort'].values, # Ch 6 - AIS fishing effort (log-norm)
], axis=-1)  # (n_months, n_lat, n_lon, 7)

y_array = features['fishing_effort'].values  # (n_months, n_lat, n_lon)

# Replace NaNs (land/masked cells) with 0 before sequence generation
X_stack = np.nan_to_num(X_stack, nan=0.0)
y_array = np.nan_to_num(y_array, nan=0.0)

print(f'X stack shape : {X_stack.shape}')   # Expected: (72, 41, 25, 7)
print(f'y array shape : {y_array.shape}')   # Expected: (72, 41, 25)


In [ ]:
# CELL: create sliding-window sequences (seq_len months in -> pred_len months ahead)
def create_sequences(X, y, seq_len, pred_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len - pred_len + 1):
        X_seq.append(X[i : i + seq_len])
        y_seq.append(y[i + seq_len : i + seq_len + pred_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_stack, y_array, SEQ_LEN, PRED_LEN)
print(f'Sequences : {X_seq.shape} -> {y_seq.shape}')
# Expected: (69, 3, 41, 25, 7) -> (69, 1, 41, 25)


In [ ]:
# CELL: chronological train / val / test split (70 / 15 / 15)
train_size = int(CONFIG['train_frac'] * len(X_seq))
val_size   = int(CONFIG['val_frac']   * len(X_seq))

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]
X_val   = X_seq[train_size : train_size + val_size]
y_val   = y_seq[train_size : train_size + val_size]
X_test  = X_seq[train_size + val_size :]
y_test  = y_seq[train_size + val_size :]

print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

# Save X_test immediately (unsqueezed, for 04_evaluation compatibility)
np.save(DATA_DIR + f['X_test_npy'], X_test)
print(f'Saved X_test -> {f["X_test_npy"]}')


In [ ]:
# CELL: reshape y arrays to match ConvLSTM output shape (N, lat, lon, 1)
y_train      = y_train.squeeze(axis=1)[..., np.newaxis]   # (N_train, 41, 25, 1)
y_val        = y_val.squeeze(axis=1)[..., np.newaxis]     # (N_val,   41, 25, 1)
y_test_model = y_test.squeeze(axis=1)[..., np.newaxis]    # (N_test,  41, 25, 1)

# Overwrite y_test with final (N, 41, 25, 1) shape --
# 04_evaluation and the dashboard both load this shape.
np.save(DATA_DIR + f['y_test_npy'], y_test_model)
print(f'Saved y_test -> {f["y_test_npy"]}  shape={y_test_model.shape}')

print(f'y_train : {y_train.shape}')
print(f'y_val   : {y_val.shape}')


In [ ]:
# CELL: build ConvLSTM2D model
# Uses keras.Input() as first layer to suppress legacy UserWarning about input_shape.
model = Sequential([
    Input(shape=(SEQ_LEN, n_lat, n_lon, N_CHANNELS)),
    ConvLSTM2D(
        filters=64, kernel_size=(3, 3),
        padding='same', return_sequences=True,
    ),
    BatchNormalization(),
    Dropout(0.2),
    ConvLSTM2D(
        filters=32, kernel_size=(3, 3),
        padding='same', return_sequences=False,
    ),
    BatchNormalization(),
    Dropout(0.2),
    Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same'),
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['mae'],
)
model.summary()


In [ ]:
# CELL: define callbacks
# ModelCheckpoint saves to .keras (native Keras format, avoids legacy HDF5 warning).
best_model_path  = DATA_DIR + f['best_model']
final_model_path = DATA_DIR + f['model_keras']

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience'],
        restore_best_weights=True,
    ),
    ModelCheckpoint(
        best_model_path,
        monitor='val_loss',
        save_best_only=True,
    ),
]
print(f'Best model checkpoint : {best_model_path}')
print(f'Final model path      : {final_model_path}')


In [ ]:
# CELL: train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1,
)

best_epoch    = int(np.argmin(history.history['val_loss'])) + 1
best_val_loss = float(np.min(history.history['val_loss']))
n_epochs_run  = len(history.history['loss'])
print(f'Training complete: {n_epochs_run} epochs, '
      f'best val_loss={best_val_loss:.6f} at epoch {best_epoch}')


In [ ]:
# CELL: save model + history + data summary
# ------------------------------------------------------------------
# 1. Final model (native .keras, no HDF5 warnings)
# ------------------------------------------------------------------
model.save(final_model_path)
print(f'Saved model   : {final_model_path}')

# ------------------------------------------------------------------
# 2. training_history.json -- read by Full_ConvLSTM.py dashboard
# ------------------------------------------------------------------
history_dict = {k: [float(v) for v in vals]
                for k, vals in history.history.items()}
with open(DATA_DIR + f['history_json'], 'w') as fh:
    json.dump(history_dict, fh, indent=2)
print(f'Saved history : {f["history_json"]}')

# ------------------------------------------------------------------
# 3. data_summary.json -- pipeline metadata for dashboard and reproducibility
# ------------------------------------------------------------------
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
summary = {
    'date_range':    f"{dm['start']} to {dm['end']}",
    'n_months':      int(n_months),
    'n_lat':         int(n_lat),
    'n_lon':         int(n_lon),
    'n_channels':    int(N_CHANNELS),
    'seq_len':       int(SEQ_LEN),
    'pred_len':      int(PRED_LEN),
    'bbox_model':    bm,
    'n_train':       int(X_train.shape[0]),
    'n_val':         int(X_val.shape[0]),
    'n_test':        int(X_test.shape[0]),
    'best_epoch':    best_epoch,
    'best_val_loss': best_val_loss,
}
with open(DATA_DIR + f['summary_json'], 'w') as fh:
    json.dump(summary, fh, indent=2)
print(f'Saved summary : {f["summary_json"]}')
print('Notebook 03 complete. Run 04_evaluation.ipynb next.')



---
## Merged Section: 04_evaluation.ipynb
---


# 04 - Model Evaluation
RMSE, MAE, F1, SSI, Wasserstein Distance

**Inputs (from Notebook 03):** `convlstm_model.keras`, `X_test.npy`, `y_test.npy`

**Outputs:** `predictions.npy`, `evaluation_results.csv`

In [ ]:
# CELL: load model and test arrays saved by Notebook 03
# Tries native .keras format first; falls back to legacy .h5 if not found.
model_keras_path = DATA_DIR + f['model_keras']
model_h5_fallback = DATA_DIR + 'convlstm_model.h5'

if os.path.exists(model_keras_path):
    model = tf.keras.models.load_model(model_keras_path)
    print(f'Loaded model  : {f["model_keras"]}')
elif os.path.exists(model_h5_fallback):
    model = tf.keras.models.load_model(model_h5_fallback)
    print(f'Loaded model  : convlstm_model.h5 (legacy fallback)')
else:
    raise FileNotFoundError(
        f'No model found at {model_keras_path} or {model_h5_fallback}. '
        'Run Notebook 03 first.'
    )

X_test = np.load(DATA_DIR + f['X_test_npy'])
y_test = np.load(DATA_DIR + f['y_test_npy'])

print(f'X_test shape  : {X_test.shape}')   # Expected: (N, 3, 41, 25, 7)
print(f'y_test shape  : {y_test.shape}')   # Expected: (N, 41, 25, 1)


In [ ]:
# CELL: generate predictions and save to predictions.npy
# Fill NaN inputs (land/coastal cells) with 0 before predicting.
X_test_clean = np.nan_to_num(X_test, nan=0.0)
preds        = model.predict(X_test_clean)

print(f'Predictions shape: {preds.shape}')   # Expected: (N, 41, 25, 1)

# Save predictions for 05_visualization.ipynb and the dashboard
np.save(DATA_DIR + f['predictions_npy'], preds)
print(f'Saved predictions -> {f["predictions_npy"]}')

# Flatten for global scalar metrics, cleaning any residual NaNs
y_flat = np.nan_to_num(y_test.flatten(),  nan=0.0)
p_flat = np.nan_to_num(preds.flatten(),   nan=0.0)
print(f'NaN in y_flat : {np.isnan(y_flat).sum()}')
print(f'NaN in p_flat : {np.isnan(p_flat).sum()}')


In [ ]:
# CELL: RMSE and MAE (global scalars over all test pixels and months)
rmse = float(np.sqrt(mean_squared_error(y_flat, p_flat)))
mae  = float(mean_absolute_error(y_flat, p_flat))
print(f'RMSE : {rmse:.6f}')
print(f'MAE  : {mae:.6f}')


In [ ]:
# CELL: F1 score (binarized at CONFIG f1_threshold = 0.5)
threshold   = CONFIG['f1_threshold']
y_binary    = (y_flat > threshold).astype(int)
pred_binary = (p_flat > threshold).astype(int)

f1 = float(f1_score(y_binary, pred_binary, zero_division=0))
print(f'F1 Score (threshold={threshold}) : {f1:.4f}')


In [ ]:
# CELL: SSI (Structural Similarity Index) -- per-sample, then averaged
def calculate_ssi(obs, pred):
    C1, C2 = 0.01**2, 0.03**2
    mu_obs, mu_pred       = obs.mean(), pred.mean()
    sigma_obs, sigma_pred = obs.std(),  pred.std()
    sigma_cross = np.mean((obs - mu_obs) * (pred - mu_pred))
    luminance = (2*mu_obs*mu_pred + C1) / (mu_obs**2 + mu_pred**2 + C1)
    contrast  = (2*sigma_obs*sigma_pred + C2) / (sigma_obs**2 + sigma_pred**2 + C2)
    structure = (sigma_cross + C2/2) / (sigma_obs*sigma_pred + C2/2)
    return luminance * contrast * structure

ssi_scores = []
for i in range(len(y_test)):
    yi = y_test[i].flatten()
    pi = preds[i].flatten()
    m  = ~(np.isnan(yi) | np.isnan(pi))
    ssi_scores.append(calculate_ssi(yi[m], pi[m]))

ssi_mean = float(np.mean(ssi_scores))
ssi_std  = float(np.std(ssi_scores))
print(f'SSI : {ssi_mean:.4f} +/- {ssi_std:.4f}')


In [ ]:
# CELL: Wasserstein Distance -- per-sample distribution comparison
wd_scores = []
for i in range(len(y_test)):
    obs_flat  = y_test[i].flatten()
    pred_flat = preds[i].flatten()
    m = ~(np.isnan(obs_flat) | np.isnan(pred_flat))
    obs_flat, pred_flat = obs_flat[m], pred_flat[m]
    obs_sum  = obs_flat.sum()
    pred_sum = pred_flat.sum()
    if obs_sum == 0 or pred_sum == 0:
        wd_scores.append(np.nan)
        continue
    wd = wasserstein_distance(
        np.arange(len(obs_flat)),
        np.arange(len(pred_flat)),
        obs_flat / obs_sum,
        pred_flat / pred_sum,
    )
    wd_scores.append(wd)

wd_valid = [s for s in wd_scores if not np.isnan(s)]
wd_mean  = float(np.mean(wd_valid))
wd_std   = float(np.std(wd_valid))
print(f'Wasserstein Distance : {wd_mean:.4f} +/- {wd_std:.4f}')


In [ ]:
# CELL: compile results table and save to evaluation_results.csv
results = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'F1', 'SSI', 'Wasserstein'],
    'Value': [
        f'{rmse:.3e}',
        f'{mae:.3e}',
        f'{f1:.4f}',
        f'{ssi_mean:.4f}',
        f'{wd_mean:.4f}',
    ],
    'Std Dev': ['N/A', 'N/A', 'N/A', f'{ssi_std:.4f}', f'{wd_std:.4f}'],
})

print('=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(results.to_string(index=False))

results.to_csv(DATA_DIR + f['eval_csv'], index=False)
print(f'Saved evaluation results -> {f["eval_csv"]}')
print('Notebook 04 complete. Run 05_visualization.ipynb next.')



---
## Merged Section: 05_visualization.ipynb
---


# 05 - Visualization
Spatial predictions, error maps, RMSE over time, training history.

**Inputs (from Notebooks 03 and 04):** `predictions.npy`, `y_test.npy`, `training_history.json`

**Outputs:** `obs_vs_pred.png`, `error_maps.png`, `rmse_over_time.png`, `training_history.png`

> **Note on latitude extent:** `bbox_model` lat range is 10-20 N, but plots use 7-20 N
> for geographic context (South China Sea / WPS coastline framing).

In [ ]:
# CELL: load predictions and test labels from Notebooks 03 and 04
y_test = np.load(DATA_DIR + f['y_test_npy'])
preds  = np.load(DATA_DIR + f['predictions_npy'])

print(f'y_test shape : {y_test.shape}')   # Expected: (N, 41, 25, 1)
print(f'preds shape  : {preds.shape}')    # Expected: (N, 41, 25, 1)

# -----------------------------------------------------------------------
# Spatial extents
# -----------------------------------------------------------------------
bm = CONFIG['bbox_model']
LON_MIN, LON_MAX = bm['lon_min'], bm['lon_max']
# bbox_model lat is 10-20 deg N; we pad the viz extent downward to 7 deg N
# for geographic context (matching the original 05_visualization layout).
LAT_MIN_VIZ, LAT_MAX_VIZ = 7, bm['lat_max']
LAT_MIN_DATA, LAT_MAX_DATA = bm['lat_min'], bm['lat_max']

print(f'Visualization extent : lat {LAT_MIN_VIZ}-{LAT_MAX_VIZ} N,  '
      f'lon {LON_MIN}-{LON_MAX} E')


In [ ]:
# CELL: define the custom fishing probability colormap
# Identical to the colormap used in Full_ConvLSTM.py dashboard.
cmap = LinearSegmentedColormap.from_list(
    'fishing',
    ['#000033', '#0000FF', '#FF00FF', '#FF0000'],
)


In [ ]:
# CELL: Figure 1 -- observed vs predicted (up to 6 test samples)
n_samples = min(6, len(y_test))
fig, axes = plt.subplots(2, n_samples, figsize=(18, 7))

for i in range(n_samples):
    # -- Observed --
    axes[0, i].imshow(
        y_test[i, :, :, 0],
        extent=[LON_MIN, LON_MAX, LAT_MIN_VIZ, LAT_MAX_VIZ],
        cmap=cmap, vmin=0, vmax=1, aspect='auto', origin='lower',
    )
    axes[0, i].set_title(f'Month {i+1}')
    if i == 0:
        axes[0, i].set_ylabel('Observed\nLatitude')

    # -- Predicted --
    im = axes[1, i].imshow(
        preds[i, :, :, 0],
        extent=[LON_MIN, LON_MAX, LAT_MIN_VIZ, LAT_MAX_VIZ],
        cmap=cmap, vmin=0, vmax=1, aspect='auto', origin='lower',
    )
    if i == 0:
        axes[1, i].set_ylabel('Predicted\nLatitude')
    axes[1, i].set_xlabel('Longitude')

fig.subplots_adjust(bottom=0.15)
cbar_ax = fig.add_axes([0.15, 0.04, 0.70, 0.03])
fig.colorbar(im, cax=cbar_ax, orientation='horizontal', label='Fishing Probability')
plt.savefig(DATA_DIR + 'obs_vs_pred.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# CELL: Figure 2 -- spatial error maps averaged across all test samples
# err = preds - y_test  (positive = model over-predicts)
err_arr  = preds - y_test                                # (N, 41, 25, 1)
mean_err = err_arr.mean(axis=0)[:, :, 0]
mae_map  = np.abs(err_arr).mean(axis=0)[:, :, 0]
rmse_map = np.sqrt((err_arr**2).mean(axis=0))[:, :, 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panels = [
    (mean_err, 'RdBu_r', -0.3,  0.3, 'Mean Error (Pred - Obs)'),
    (mae_map,  'Reds',    0.0,  0.3, 'MAE per cell'),
    (rmse_map, 'Reds',    0.0,  0.3, 'RMSE per cell'),
]
for ax, (arr, cmap_name, vmin, vmax, title) in zip(axes, panels):
    im = ax.imshow(
        arr,
        extent=[LON_MIN, LON_MAX, LAT_MIN_VIZ, LAT_MAX_VIZ],
        cmap=cmap_name, vmin=vmin, vmax=vmax,
        aspect='auto', origin='lower',
    )
    ax.set_title(title)
    ax.set_xlabel('Longitude (E)')
    ax.set_ylabel('Latitude (N)')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(DATA_DIR + 'error_maps.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# CELL: Figure 3 -- per-sample RMSE over the test period
from sklearn.metrics import mean_squared_error

rmse_list = [
    float(np.sqrt(mean_squared_error(
        np.nan_to_num(y_test[i].flatten(), nan=0.0),
        np.nan_to_num(preds[i].flatten(),  nan=0.0),
    )))
    for i in range(len(y_test))
]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(range(len(rmse_list)), rmse_list, alpha=0.7, color='steelblue')
ax.plot(range(len(rmse_list)), rmse_list,
        color='navy', linewidth=1.5, marker='o', markersize=5)
ax.axhline(float(np.mean(rmse_list)), color='crimson', linewidth=1.2,
           linestyle='--', label=f'mean RMSE = {np.mean(rmse_list):.4f}')
ax.set_xlabel('Test sample index')
ax.set_ylabel('RMSE')
ax.set_title('Per-sample RMSE over the test period')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR + 'rmse_over_time.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# CELL: Figure 4 -- training history curves
import json as _json

history_path = DATA_DIR + f['history_json']
if not os.path.exists(history_path):
    print(f'training_history.json not found at {history_path}.')
    print('Run 03_model_training.ipynb first.')
else:
    with open(history_path) as fh:
        history_dict = _json.load(fh)

    loss_tr = history_dict.get('loss', [])
    loss_vl = history_dict.get('val_loss', [])
    mae_tr  = history_dict.get('mae', history_dict.get('mean_absolute_error', []))
    mae_vl  = history_dict.get('val_mae', history_dict.get('val_mean_absolute_error', []))
    epochs  = list(range(1, len(loss_tr) + 1))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, loss_tr, label='Train loss')
    ax1.plot(epochs[:len(loss_vl)], loss_vl, linestyle='--', label='Val loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Binary cross-entropy')
    ax1.set_title('Training Loss'); ax1.legend()

    ax2.plot(epochs, mae_tr, label='Train MAE')
    ax2.plot(epochs[:len(mae_vl)], mae_vl, linestyle='--', label='Val MAE')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE')
    ax2.set_title('Training MAE'); ax2.legend()

    plt.tight_layout()
    plt.savefig(DATA_DIR + 'training_history.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('Notebook 05 complete.')
